# Notebook 20 — The FIRST official SCV-IDS benchmark comparison

**Purpose:** Execute the *single preselected* 444-feature rare-class-weighted LightGBM methodology on the original UNSW-NB15 official train/test partition. This is the publication-critical evaluation. It is **not another internal-validation experiment**.

## What was frozen *before viewing official test labels*

- Model: the E3 rare-weighted **444-feature** LightGBM with five serialized rows, 15 native categorical fields, and 234 numeric within-window statistical features. No new architecture or prediction-correction gate.
- Full official train: **175,341 original records → 175,337 labeled five-row windows**; test: **82,332 original records → 82,328 scored windows**, separately constructed within each split.
- Learning rate **0.06**, **63 leaves**, **minimum 60 leaves data**, **sqrt inverse-frequency weights capped at 12**, random seed **42**.
- **Exactly 58 boosting rounds**, the median of five E3/E4 internal-validation best iterations `[74, 58, 61, 36, 48]`; **no training, tuning, or early stopping on official test**.
- Native categorical vocabulary fitted **only** on full training partition. The same order and feature manifest as the completed E3 reproduction, with SHA-256 verified official training source.
- Paper's reported multiclass 85.77% accuracy, 85.16% aggregate precision, 85.77% aggregate recall, 84.59% aggregate F1, and separately 46.87% macro-F1. The paper's aggregate F1 averaging method is not presumed weighted: we will independently publish macro, weighted and micro precision/recall/F1.

## Tamper-evident evidence and crash protection

1. **Before opening the official testing CSV**, fit full-training model and freeze booster, full config, feature schema, original training SHA256, and the fitted model SHA256.
2. Automatically upload this original booster and its metadata in an **unpublished GitHub Release**; commit a *public training-only metadata proof* recording model SHA256. This establishes source-level training/model freeze before test access.
3. **Only then** read testing rows and predict once. Compute per-class precision/recall/F1, ten-class confusion, accuracy, macro/micro/weighted precision/recall/F1, binary attack detection, and explicit comparisons to reported SCV numbers.
4. Independently verify the genuine saved per-row test predictions and confusion, archive raw predictions **privately in a second unpublished release**, commit only aggregate metrics. A second rerun is prohibited once outputs show official test was evaluated.

**Critical integrity notes:** The test CSV row ordering is not proven actual chronology. Matching the official split/window size is substantially better than internal-only comparison but is not identical to all SCV-IDS preprocessing/training settings. The test should NOT be reused later for model tuning. Whatever metrics appear, report them, including negative findings. If a fresh Kaggle environment already contains a published official evaluation in GitHub, do not launch a second test.

## Kaggle setup

**CPU only (Accelerator None)**; **Internet ON**; authorize existing private GitHub secret **`pushRare`**. Attach a Kaggle UNSW-NB15 dataset that includes **BOTH** official CSVs `UNSW_NB15_training-set.csv` and `UNSW_NB15_testing-set.csv` (ideally the same dataset; not raw four-file dumps). Do not attach multiple copies of either filename. Then choose **Save Version → Save & Run All**. The script preflights dataset availability and must not accidentally read the test before pretest model freeze.

**Final log marker:** `OFFICIAL-001 FINAL TEST COMPLETED, PRIVATE ARCHIVED AND VERIFIED`.

In [ ]:
# ONE CELL — irreversible first official train/test evaluation; pre-test public freeze.
import os,sys,subprocess,hashlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-official-001")
OUT=Path("/kaggle/working/research_outputs/WINDOW-OFFICIAL-001-S42")
NAME_TRAIN="UNSW_NB15_training-set.csv"
NAME_TEST="UNSW_NB15_testing-set.csv"
ROOT=Path("/kaggle/input")
training=list(ROOT.rglob(NAME_TRAIN))
testing=list(ROOT.rglob(NAME_TEST))
print("Official source TRAIN:",[str(p) for p in training],flush=True)
print("Official source TEST :",[str(p) for p in testing],flush=True)
assert len(training)==1 and len(testing)==1,(
    "Attach a single complete official UNSW-NB15 Kaggle dataset with BOTH "
    "training-set.csv and testing-set.csv, and remove duplicate attachments."
)
with training[0].open("rb") as f:
    assert hashlib.file_digest(f,"sha256").hexdigest()==(
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
    ),"Training source SHA256 differs from frozen dataset: stop."
assert not (OUT/"official_evaluation_complete.json").exists(),(
    "Official test was already evaluated in this environment; refuse a second test."
)
token=UserSecretsClient().get_secret("pushRare")
assert token,"Authorize Kaggle secret pushRare for pretest and final private GitHub records"
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
source_sha=subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
print("OFFICIAL EXPERIMENT SOURCE CODE:",source_sha,flush=True)
env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["GITHUB_TOKEN"]=token
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
subprocess.run([
    sys.executable,str(REPO/"scripts/run_official_001.py"),
    "--repo",str(REPO),"--config",str(REPO/"configs/window_official_001_frozen.json"),
    "--input",str(ROOT),"--output",str(OUT),"--auto-archive"
],cwd=str(REPO),env=env,check=True)
print("OFFICIAL TEST FULL METRICS (one frozen model):",flush=True)
print((OUT/"official_metrics.json").read_text(),flush=True)
print("OFFICIAL BENCHMARK COMPARISON:",flush=True)
print((OUT/"paper_comparison.json").read_text(),flush=True)
print("Done. Do NOT rerun this test for tuning.",flush=True)
